# Final chronological held-out validation: non-seasonal versus seasonal Poisson model

This notebook is the leakage-safe model comparison. The first 60 weekly observations are used once to fit each model. Those global parameter vectors are then frozen. The remaining observations are divided into ten non-overlapping six-week outcome periods. At each later origin, only the four most recent observations available on that date update the hidden $E$ and $I$ state; the following six weeks remain unseen until after the forecast is generated.

Both models therefore forecast exactly the same dates from exactly the same information. The notebook reports all ten chronological periods and a pre-selected balanced subset containing four outbreak and four non-outbreak periods. The balanced subset is used for confusion-matrix comparison; it is selected from observed outcomes before model forecasts are inspected.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'outbreak_probability_model': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

from outbreak_probability_model.model import load_default_inputs
from outbreak_probability_model.london_calibration import (
    CalibrationConfig, FIT_PARAMETER_BOUNDS, SEASONAL_FIT_PARAMETER_BOUNDS,
    HistoryConditioningConfig, binary_classification_metrics_across_cutoffs,
    calibration_starting_vector, default_calibration_parameters,
    load_london_confirmed_cases, make_balanced_blocked_validation_design,
    strict_forecast_validation,
)

OUTPUT = ROOT / 'experiments' / 'measles' / 'London' / 'calibration_strict_validation_final'
OUTPUT.mkdir(parents=True, exist_ok=True)
QUICK_CHECK = False
TRAINING_WEEKS = 60
HORIZON_WEEKS = 6
OUTBREAK_THRESHOLD = 15.0
ALARM_PROBABILITY_CUTOFF = 0.60
N_STOCHASTIC_SIMULATIONS = 30 if QUICK_CHECK else 1000
config = CalibrationConfig(
    block_weeks=6, n_trials=20 if QUICK_CHECK else 120,
    n_refinement_trials=6 if QUICK_CHECK else 40,
    initial_state_refinement_maxiter=0 if QUICK_CHECK else 50,
    seasonal_refinement_maxiter=4 if QUICK_CHECK else 70,
    warmup_weeks=0, random_seed=20260825,
)
history_conditioning = HistoryConditioningConfig(
    history_weeks=4, transmission_multiplier_bounds=(0.8, 1.25),
    regularization_strength=1.0, origin_observation_weight=4.0, maxiter=80,
)
print('Output directory:', OUTPUT)
print('Stochastic paths per model and origin:', N_STOCHASTIC_SIMULATIONS)

## 1. Fixed chronological design

Sixty training weeks provide more than one full annual cycle before seasonality is estimated. The later ten outcome blocks do not overlap. The same design and random seed are used for both models.

In [ ]:
cases = load_london_confirmed_cases()
inputs = load_default_inputs()
base = default_calibration_parameters()
design = make_balanced_blocked_validation_design(
    cases, training_weeks=TRAINING_WEEKS, horizon_weeks=HORIZON_WEEKS,
    outbreak_threshold=OUTBREAK_THRESHOLD, random_seed=config.random_seed,
)
cutoff_indices = design.cutoff_index.astype(int).tolist()
design.to_csv(OUTPUT / '00_validation_design.csv', index=False)
display(design)
display(design.groupby(['selected_for_balanced_test', 'observed_event']).size().rename('blocks'))

## 2. Fit once on the training period and forecast all later blocks

These calls do **not** load the all-data fitted CSV files. Each model is fitted independently using only the first 60 observations, preventing information from the held-out period entering the comparison. `refit_parameters_each_cutoff=False` freezes that training-period vector at every later origin.

In [ ]:
nonseasonal_start = calibration_starting_vector(base, FIT_PARAMETER_BOUNDS)
seasonal_start = calibration_starting_vector(base, SEASONAL_FIT_PARAMETER_BOUNDS)
seasonal_start.update(seasonal_amplitude=0.15, seasonal_peak_week=20.0)

runs = {}
for model_name, bounds, start in [
    ('non-seasonal', FIT_PARAMETER_BOUNDS, nonseasonal_start),
    ('seasonal', SEASONAL_FIT_PARAMETER_BOUNDS, seasonal_start),
]:
    print('\nRunning', model_name)
    summary, paths, fits = strict_forecast_validation(
        cases, inputs=inputs, config=config, cutoff_indices=cutoff_indices,
        horizon_weeks=HORIZON_WEEKS,
        n_stochastic_simulations=N_STOCHASTIC_SIMULATIONS,
        outbreak_threshold=OUTBREAK_THRESHOLD, base_parameters=base,
        history_conditioning=history_conditioning,
        refit_parameters_each_cutoff=False, objective_metric='poisson_nll',
        parameter_bounds=bounds, starting_vector=start, progress=True,
    )
    summary.insert(0, 'model', model_name)
    paths.insert(0, 'model', model_name)
    fits.insert(0, 'model', model_name)
    runs[model_name] = (summary, paths, fits)

all_summary = pd.concat([value[0] for value in runs.values()], ignore_index=True)
all_paths = pd.concat([value[1] for value in runs.values()], ignore_index=True)
all_fits = pd.concat([value[2] for value in runs.values()], ignore_index=True)
all_summary.to_csv(OUTPUT / '01_weekly_forecast_summary.csv', index=False)
all_paths.to_csv(OUTPUT / '02_stochastic_forecast_paths.csv', index=False)
all_fits.to_csv(OUTPUT / '03_training_fit_and_origin_states.csv', index=False)

## 3. Held-out metrics and confusion matrices

In [ ]:
origin_results = all_fits.merge(
    design[['cutoff_index', 'block_id', 'selected_for_balanced_test',
            'observed_event', 'forecast_start', 'forecast_end']],
    on='cutoff_index', how='left', validate='many_to_one',
)
origin_results = origin_results.rename(columns={
    'predicted_six_week_event_probability': 'predicted_probability',
})
origin_results.to_csv(OUTPUT / '04_origin_probability_results.csv', index=False)

metric_rows = []
classification_rows = []
for model_name, weekly in all_summary.groupby('model'):
    origin = origin_results.query('model == @model_name').copy()
    subsets = [
        ('all chronological blocks', np.ones(len(origin), dtype=bool)),
        ('balanced test blocks', origin.selected_for_balanced_test.to_numpy(bool)),
    ]
    for subset_name, keep in subsets:
        selected_origin = origin.loc[keep]
        selected_weekly = weekly[weekly.cutoff_id.isin(selected_origin.cutoff_id)]
        residual = selected_weekly.median_cases - selected_weekly.observed_cases
        metric_rows.append({
            'model': model_name, 'subset': subset_name, 'origins': len(selected_origin),
            'weekly_mae': residual.abs().mean(),
            'weekly_rmse': np.sqrt(np.mean(residual ** 2)),
            'p10_p90_coverage': np.mean(
                (selected_weekly.observed_cases >= selected_weekly.p10_cases) &
                (selected_weekly.observed_cases <= selected_weekly.p90_cases)
            ),
            'brier_score': np.mean(
                (selected_origin.predicted_probability -
                 selected_origin.observed_event.astype(float)) ** 2
            ),
        })
        table = binary_classification_metrics_across_cutoffs(
            selected_origin, probability_column='predicted_probability',
            outcome_column='observed_event',
            probability_cutoffs=[ALARM_PROBABILITY_CUTOFF],
        )
        table.insert(0, 'subset', subset_name)
        table.insert(0, 'model', model_name)
        classification_rows.append(table)
metrics = pd.DataFrame(metric_rows)
classification = pd.concat(classification_rows, ignore_index=True)
metrics.to_csv(OUTPUT / '05_held_out_metrics.csv', index=False)
classification.to_csv(OUTPUT / '06_classification_metrics.csv', index=False)
display(metrics)
display(classification)

In [ ]:
balanced = classification.query("subset == 'balanced test blocks'")
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, row in zip(axes, balanced.itertuples()):
    matrix = np.array([[row.true_negative, row.false_positive],
                       [row.false_negative, row.true_positive]])
    ax.imshow(matrix, cmap='Blues', vmin=0, vmax=max(1, matrix.max()))
    for (i, j), value in np.ndenumerate(matrix):
        ax.text(j, i, str(value), ha='center', va='center', fontsize=14)
    ax.set_xticks([0, 1], ['no alarm', 'alarm'])
    ax.set_yticks([0, 1], ['no outbreak', 'outbreak'])
    ax.set(title=f'{row.model}\ncut-off = {ALARM_PROBABILITY_CUTOFF:.0%}',
           xlabel='Forecast decision', ylabel='Observed outcome')
fig.suptitle('Held-out balanced-test confusion matrices')
fig.tight_layout()
fig.savefig(OUTPUT / '07_balanced_confusion_matrices.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. Chronological probability audit

Red markers are observed outbreak periods and green markers are non-outbreak periods. The horizontal line is the alarm cut-off.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, sharey=True)
for ax, (model_name, data) in zip(axes, origin_results.groupby('model', sort=False)):
    data = data.sort_values('forecast_start')
    colours = np.where(data.observed_event, '#c62828', '#2e7d32')
    ax.scatter(data.forecast_start, data.predicted_probability, c=colours, s=65, zorder=3)
    ax.plot(data.forecast_start, data.predicted_probability, color='0.55', lw=1)
    ax.axhline(ALARM_PROBABILITY_CUTOFF, color='black', ls='--', lw=1)
    ax.set(title=model_name, ylabel='Forecast outbreak probability', ylim=(-.03, 1.03))
    ax.grid(alpha=.25)
axes[-1].set_xlabel('Start of held-out six-week period')
fig.suptitle('Chronological held-out outbreak-probability forecasts')
fig.tight_layout()
fig.savefig(OUTPUT / '08_chronological_probability_audit.png', dpi=180, bbox_inches='tight')
plt.show()

# Results-ready held-out forecast pages with uncertainty and classification.
classified_origins = origin_results.copy()
classified_origins['predicted_alarm'] = (
    classified_origins.predicted_probability >= ALARM_PROBABILITY_CUTOFF
)
classified_origins['classification'] = np.select(
    [
        classified_origins.observed_event & classified_origins.predicted_alarm,
        classified_origins.observed_event & ~classified_origins.predicted_alarm,
        ~classified_origins.observed_event & classified_origins.predicted_alarm,
    ],
    ['DETECTED OUTBREAK', 'MISSED OUTBREAK', 'FALSE ALARM'],
    default='CORRECT BELOW THRESHOLD',
)
classification_colours = {
    'DETECTED OUTBREAK': '#238b45', 'MISSED OUTBREAK': '#c51b1d',
    'FALSE ALARM': '#d97706', 'CORRECT BELOW THRESHOLD': '#4c78a8',
}
page_dir = OUTPUT / 'held_out_classified_forecast_pages'
page_dir.mkdir(parents=True, exist_ok=True)
page_pdf = OUTPUT / '09_held_out_classified_forecasts.pdf'
with PdfPages(page_pdf) as pdf:
    for cutoff_id in sorted(all_summary.cutoff_id.unique()):
        fig, axes = plt.subplots(2, 1, figsize=(10.5, 7.2), sharex=True, sharey=True)
        origin_date = None
        for ax, model_name in zip(axes, ('non-seasonal', 'seasonal')):
            weekly = all_summary.query(
                'cutoff_id == @cutoff_id and model == @model_name'
            ).sort_values('forecast_week')
            origin = classified_origins.query(
                'cutoff_id == @cutoff_id and model == @model_name'
            ).iloc[0]
            origin_date = pd.Timestamp(origin.forecast_start)
            colour = classification_colours[origin.classification]
            x = weekly.forecast_week.to_numpy(float)
            ax.fill_between(x, weekly.p10_cases.to_numpy(float),
                            weekly.p90_cases.to_numpy(float),
                            color='#8ecae6', alpha=.32, label='p10-p90')
            ax.plot(x, weekly.median_cases.to_numpy(float),
                    color='#023047', lw=2.0,
                    label='stochastic median')
            ax.plot(x, weekly.observed_cases.to_numpy(float),
                    'ko-', lw=1.3, ms=4.0,
                    label='withheld observed cases')
            ax.axhline(OUTBREAK_THRESHOLD, color='#6a1b1a', ls='--', lw=1.2,
                       label='outbreak threshold')
            ax.set_title(
                f'{model_name.capitalize()}: {origin.classification}; '
                f'probability={origin.predicted_probability:.1%}',
                color=colour, fontweight='bold',
            )
            ax.set_ylabel('Reported cases')
            ax.grid(alpha=.22)
        axes[-1].set_xlabel('Forecast week')
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', ncol=4, frameon=False)
        fig.suptitle(f'Held-out six-week forecast from {origin_date.date()}', y=.995)
        fig.tight_layout(rect=(0, 0, 1, .93))
        pdf.savefig(fig, bbox_inches='tight')
        fig.savefig(page_dir / f'origin_{cutoff_id:02d}_{origin_date:%Y%m%d}.png',
                    dpi=180, bbox_inches='tight')
        plt.close(fig)
print('Saved held-out classified forecast PDF:', page_pdf)
print('Saved held-out classified forecast PNGs:', page_dir)

## Interpretation

Use the balanced-test Brier score, sensitivity, specificity and confusion matrices for the primary seasonal-versus-non-seasonal comparison. Report the all-block metrics as a secondary check because they preserve every chronological outcome block. The all-data calibration notebooks are still used to obtain the final parameter vector after model selection; this notebook tests whether the modelling choice generalises beyond its training period.